# Population-weighted density → ITL2 (UK, 2011–2024)

Builds **population-weighted density (PWD)** per ITL2 region per year, on **both ITL2 2025** (`clean/`) **and ITL2 2021** (`clean/itl2_2021/`) boundaries: `pop_weighted_density_itl2.csv`.

**What it measures.** Plain density (`population_density_itl2.csv`) is people ÷ land area — the region's *land per person*, dominated by empty land in rural regions. **PWD is the density of the neighbourhood the average resident lives in**:

$$\text{PWD}_r = \frac{\sum_{i \in r} p_i \, d_i}{\sum_{i \in r} p_i}, \qquad d_i = \frac{p_i}{\text{land area}_i}$$

over small areas *i*: **England & Wales** 2021 LSOAs (~1,500 residents each; 35,672), **Scotland** 2022 Data Zones (~750 residents; 7,392), **Northern Ireland** 2021 Data Zones (~500 residents; 3,780). It is the standard "experienced density" measure in urban economics (e.g. Ciccone & Hall 1996; Glaeser & Kahn 2004) and is less sensitive to how much uninhabited land a region contains. The ratio `pwd_to_area_density` (PWD ÷ plain density) measures how **concentrated** settlement is within the region.

**Sources (fetched in code; raw copies in `raw/density/`):**
- E&W population: Nomis **NM_2014_1** (ONS small-area estimates, 2021-based), 2011–2024. Land area: ONS *Standard Area Measurements for 2021 Statistical Geographies* (land only), which also gives each LSOA's LA.
- Scotland population: NRS *Small area population estimates, mid-2011 to mid-2024, 2022 Data Zones* (one consistent series; each file gives the DZ's council area). Area: Scottish Government *Data Zone Boundaries 2022* (`stdareakm2`).
- NI population: NISRA *Mid-2024 population estimates — Data Zones* (2021–2024 only). Area: NISRA *Data Zone 2021 boundaries* (`Area_ha`; also gives each DZ's council).

Coverage: **E&W and Scotland 2011–2024; NI 2021–2024** (NI's 2021 Data Zones only exist from 2021; earlier NI small areas are a different geography).

**Caveats:** small areas differ in typical size between nations (LSOA ~1,500, Scottish DZ ~750, NI DZ ~500 residents), and PWD rises as units get smaller — so **compare PWD within a nation over time, and across regions with care across national borders**. Scottish/NI areas are standard areas (may include inland water); E&W is land only. LSOA populations sum exactly to the LA mid-year estimates for 2011–2021; for 2022–2024 the small-area estimates predate ONS's July 2026 revision (international migration), so they differ slightly from `population_itl2.csv` (median ~0.1–0.2% by LA; up to ~6% for City of London / Westminster). The value depends on the size of the small areas used (LSOAs vary in area), so compare PWD across regions and over time, not with PWD built from other geographies.

In [1]:
import io, time, urllib.request, zipfile
from datetime import date
from pathlib import Path
import numpy as np
import pandas as pd

BASE = Path("/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel")
RAW_DIR   = BASE / "raw" / "density"
CROSSWALK = BASE / "lib" / "la_code_crosswalk.csv"
POP_LA    = sorted((BASE / "raw" / "population").glob("nomis_population_raw_*.csv"))[-1]
SPINES = {"2025": (BASE / "clean" / "geography_spine.csv",               BASE / "clean"),
          "2021": (BASE / "clean" / "itl2_2021" / "geography_spine.csv", BASE / "clean" / "itl2_2021")}

URL_POP = ("https://www.nomisweb.co.uk/api/v01/dataset/NM_2014_1.data.csv?geography=TYPE151"
           "&gender=0&c_age=200&measures=20100&date=first-latest&select=date,geography_code,obs_value")
URL_SAM = "https://www.arcgis.com/sharing/rest/content/items/a488cb8fc9a74accb63cb52961e456ef/data"
SAM_ZIP = RAW_DIR / "SAM_2021_statistical_geographies_EW.zip"
URL_SCOT_POP = "https://nrscotland.gov.uk/media/yqqbrvbw/small-area-population-estimates-mid-2011-to-mid-2024-2022-dz.zip"
URL_SCOT_DZ  = "https://maps.gov.scot/ATOM/shapefiles/SG_DataZoneBdry_2022.zip"
URL_NI_POP   = "https://www.nisra.gov.uk/system/files/statistics/2025-12/MYE24_DZ.xlsx"
URL_NI_DZ    = "https://www.nisra.gov.uk/files/nisra/publications/geography-dz2021-esri-shapefile.zip"
DOWNLOADS = {URL_SCOT_POP: "nrs_sape_2011_2024_dz2022.zip", URL_SCOT_DZ: "SG_DataZoneBdry_2022.zip",
             URL_NI_POP: "nisra_MYE24_DZ.xlsx", URL_NI_DZ: "nisra_dz2021_shapefile.zip"}
for u in [URL_POP, URL_SAM, *DOWNLOADS]:
    print(u)

remap = dict(zip(*[pd.read_csv(CROSSWALK, dtype=str)[c] for c in ("old_code", "new_code")]))
RAW_DIR.mkdir(parents=True, exist_ok=True)
for u, f in DOWNLOADS.items():                     # download once
    if not (RAW_DIR / f).exists():
        req = urllib.request.Request(u, headers={"User-Agent": "Mozilla/5.0"})
        (RAW_DIR / f).write_bytes(urllib.request.urlopen(req, timeout=600).read())

https://www.nomisweb.co.uk/api/v01/dataset/NM_2014_1.data.csv?geography=TYPE151&gender=0&c_age=200&measures=20100&date=first-latest&select=date,geography_code,obs_value
https://www.arcgis.com/sharing/rest/content/items/a488cb8fc9a74accb63cb52961e456ef/data
https://nrscotland.gov.uk/media/yqqbrvbw/small-area-population-estimates-mid-2011-to-mid-2024-2022-dz.zip
https://maps.gov.scot/ATOM/shapefiles/SG_DataZoneBdry_2022.zip
https://www.nisra.gov.uk/system/files/statistics/2025-12/MYE24_DZ.xlsx
https://www.nisra.gov.uk/files/nisra/publications/geography-dz2021-esri-shapefile.zip


## 1. LSOA inputs

Population is paged past Nomis's 25,000-row cap (~500k rows); raw copies are saved, dated, in `raw/density/`.

In [2]:
def nomis_all(url, limit=25000):
    frames, offset = [], 0
    while True:
        for k in range(3):
            try:
                with urllib.request.urlopen(f"{url}&recordlimit={limit}&recordoffset={offset}", timeout=300) as r:
                    part = pd.read_csv(io.BytesIO(r.read())); break
            except Exception:
                if k == 2: raise
                time.sleep(5)
        frames.append(part)
        if len(part) < limit: return pd.concat(frames, ignore_index=True)
        offset += limit

pop = nomis_all(URL_POP)
assert not pop.duplicated(["DATE", "GEOGRAPHY_CODE"]).any()
pop.to_csv(RAW_DIR / f"nomis_lsoa_population_{date.today().isoformat()}.csv", index=False)
pop = pop.rename(columns={"DATE": "year", "GEOGRAPHY_CODE": "lsoa", "OBS_VALUE": "population"})
print(f"LSOA population: {pop.lsoa.nunique():,} LSOAs x {pop.year.nunique()} years ({pop.year.min()}-{pop.year.max()})")

if not SAM_ZIP.exists():
    SAM_ZIP.write_bytes(urllib.request.urlopen(URL_SAM, timeout=300).read())
z = zipfile.ZipFile(SAM_ZIP)
sam = pd.read_csv(z.open(next(n for n in z.namelist() if n.endswith("SAM_LSOA_DEC_2021_EW.csv"))), encoding="latin-1")
sam = sam.rename(columns={"LSOA21CD": "lsoa", "Land count (Area in Hectares)": "land_ha", "LTLA22CD": "la_code"})
sam["la_code"] = sam.la_code.replace(remap)
sam["land_km2"] = sam.land_ha / 100
print(f"LSOA land area: {len(sam):,} LSOAs, {sam.land_km2.sum():,.0f} km2 | zero-land LSOAs: {(sam.land_km2 <= 0).sum()}")

lsoa = pop.merge(sam[["lsoa", "la_code", "land_km2"]], on="lsoa", how="left")
assert lsoa.land_km2.notna().all(), "LSOA without a land area"
lsoa["area_type"] = "LSOA 2021 (E&W)"

# --- Scotland: 2022 Data Zones, mid-2011 to mid-2024 ---
import geopandas as gpd
zs = zipfile.ZipFile(RAW_DIR / "nrs_sape_2011_2024_dz2022.zip")
scot = []
for n in sorted(zs.namelist()):
    if not n.endswith(".xlsx"): continue
    yr = int("".join(ch for ch in n if ch.isdigit())[:4])
    d = pd.read_excel(zs.open(n), sheet_name=str(yr), header=2, usecols=[0, 2, 4, 5])
    d.columns = ["lsoa", "la_code", "sex", "population"]
    scot.append(d[d.sex.astype(str).str.strip().eq("Persons")].assign(year=yr).drop(columns="sex"))
scot = pd.concat(scot, ignore_index=True)
dz = gpd.read_file(f"zip://{RAW_DIR / 'SG_DataZoneBdry_2022.zip'}!SG_DataZone_Bdry_2022.shp", ignore_geometry=True)
scot = scot.merge(dz[["dzcode", "stdareakm2"]].rename(columns={"dzcode": "lsoa", "stdareakm2": "land_km2"}), on="lsoa", how="left")
assert scot.land_km2.notna().all() and not scot.duplicated(["lsoa", "year"]).any()
scot["area_type"] = "Data Zone 2022 (Scotland)"
print(f"Scotland: {scot.lsoa.nunique():,} data zones x {scot.year.nunique()} years ({scot.year.min()}-{scot.year.max()})")

# --- Northern Ireland: 2021 Data Zones, mid-2021 to mid-2024 ---
ni = pd.read_excel(RAW_DIR / "nisra_MYE24_DZ.xlsx", sheet_name="Flat")
ni = ni.rename(columns={"Area_Code": "lsoa", "Year": "year", "MYE": "population"})[["lsoa", "year", "population"]]
nz = gpd.read_file(f"zip://{RAW_DIR / 'nisra_dz2021_shapefile.zip'}!DZ2021.shp", ignore_geometry=True)
ni = ni.merge(nz[["DZ2021_cd", "LGD2014_cd", "Area_ha"]].rename(columns={"DZ2021_cd": "lsoa", "LGD2014_cd": "la_code"}), on="lsoa", how="left")
assert ni.Area_ha.notna().all() and not ni.duplicated(["lsoa", "year"]).any()
ni = ni.assign(land_km2=ni.Area_ha / 100, area_type="Data Zone 2021 (NI)").drop(columns="Area_ha")
print(f"NI: {ni.lsoa.nunique():,} data zones x {ni.year.nunique()} years ({ni.year.min()}-{ni.year.max()})")

lsoa = pd.concat([lsoa, scot, ni], ignore_index=True)
lsoa["la_code"] = lsoa.la_code.replace(remap)
lsoa["density"] = lsoa.population / lsoa.land_km2

# check: LSOAs reproduce LA populations (Nomis LA mid-year estimates)
la_pop = pd.read_csv(POP_LA).assign(la_code=lambda d: d.GEOGRAPHY_CODE.replace(remap)).groupby(["la_code", "DATE"]).OBS_VALUE.sum()
chk = lsoa.groupby(["la_code", "year"]).population.sum().to_frame().join(la_pop.rename_axis(["la_code", "year"]).rename("la_mye"))
gap = (100 * (chk.population / chk.la_mye - 1)).abs()
gap = gap.reset_index()
gap["nation"] = gap.la_code.str[0].map({"E": "E&W", "W": "E&W", "S": "Scotland", "N": "NI"})
print("Small-area sums vs LA mid-year estimates, |gap| %:")
print(gap.groupby("nation")[0].agg(median="median", max="max").round(3).to_string())

LSOA population: 35,672 LSOAs x 14 years (2011-2024)
LSOA land area: 35,672 LSOAs, 151,047 km2 | zero-land LSOAs: 0


Scotland: 7,392 data zones x 14 years (2011-2024)


NI: 3,780 data zones x 4 years (2021-2024)


Small-area sums vs LA mid-year estimates, |gap| %:
          median    max
nation                 
E&W        0.000  6.256
NI         0.002  0.020
Scotland   0.000  0.000


## 2. PWD by ITL2 (both boundary versions)

Per region-year: population, land area, plain density, PWD, and the concentration ratio PWD ÷ plain density.

In [3]:
for vintage, (spine_path, out_dir) in SPINES.items():
    spine = pd.read_csv(spine_path)
    to_itl2 = dict(zip(spine.la_code, spine.itl2_code)); names = dict(zip(spine.itl2_code, spine.itl2_name))
    d = lsoa.assign(itl2_code=lsoa.la_code.map(to_itl2))
    assert d.itl2_code.notna().all(), f"LSOA LA not in the {vintage} spine"
    d["p_x_d"] = d.population * d.density
    g = d.groupby(["itl2_code", "year"]).agg(population=("population", "sum"), land_area_km2=("land_km2", "sum"),
                                             p_x_d=("p_x_d", "sum"), small_area_type=("area_type", "first")).reset_index()
    g["area_density"]        = g.population / g.land_area_km2
    g["pop_weighted_density"] = g.p_x_d / g.population
    g["pwd_to_area_density"]  = g.pop_weighted_density / g.area_density
    g.insert(1, "itl2_name", g.itl2_code.map(names))
    g = g.drop(columns="p_x_d").sort_values(["year", "itl2_code"]).reset_index(drop=True)
    g.to_csv(out_dir / "pop_weighted_density_itl2.csv", index=False, float_format="%.4f")
    print(f"\nITL2 {vintage}: {g.itl2_code.nunique()} regions x {g.year.nunique()} years -> {out_dir.name}/pop_weighted_density_itl2.csv")

    last = g[g.year == g.year.max()].set_index("itl2_name")
    cols = ["area_density", "pop_weighted_density", "pwd_to_area_density"]
    print(f"  {g.year.max()}: highest / lowest PWD (people per km2):")
    print(pd.concat([last.nlargest(3, "pop_weighted_density"), last.nsmallest(3, "pop_weighted_density")])[cols].round(1).to_string())
    print("  most concentrated (PWD / area density):")
    print(last.nlargest(3, "pwd_to_area_density")[cols].round(1).to_string())


ITL2 2025: 46 regions x 14 years -> clean/pop_weighted_density_itl2.csv
  2024: highest / lowest PWD (people per km2):
                                    area_density  pop_weighted_density  pwd_to_area_density
itl2_name                                                                                  
Inner London - East                      11376.8               15412.7                  1.4
Inner London - West                      10214.9               14631.6                  1.4
Outer London - East and North East        4610.7                8577.1                  1.9
North Wales                                113.3                1448.0                 12.8
Highlands and Islands                       12.2                1531.8                125.8
Cornwall and Isles of Scilly               164.4                1606.6                  9.8
  most concentrated (PWD / area density):
                        area_density  pop_weighted_density  pwd_to_area_density
itl2_name             